In [ ]:
# !pip install simple-icd-10 simple-icd-10-cm --quiet

In [19]:
import os
import re
import numpy as np
import pandas as pd
from IPython.display import display
import simple_icd_10 as icd
import simple_icd_10_cm as icd_cm
from functools import partial

In [2]:
df = pd.read_csv("trajectories.csv")
df.shape

(5938215, 9)

In [3]:
df.head()

,Unnamed: 0,subject_id,hadm_id,seq_num,icd10_code,icd10_category,admittime,dischtime,deathtime
0,0,10000032,22595853,1,K766,K76,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN
1,1,10000032,22595853,2,R188,R18,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN
2,2,10000032,22595853,3,K740,K74,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN
3,3,10000032,22595853,4,B1920,B19,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN
4,4,10000032,22595853,5,J449,J44,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN


# Getting descriptions of ICD10 codes

In [8]:
categories = df.groupby(by="icd10_category").agg({"icd10_code": "count"}).reset_index().rename(columns={"icd10_code": "count"})

In [10]:
categories.head()

,icd10_category,count
0,A01,13
1,A02,137
2,A03,36
3,A04,7034
4,A05,78


In [22]:
def get_description(icd10_category, lib=icd):
    try:
        return lib.get_description(icd10_category)
    except:
        return None

In [ ]:
categories["description"] = categories["icd10_category"].apply(partial(get_description, lib=icd))

In [12]:
categories.head()

,icd10_category,count,description
0,A01,13,Typhoid and paratyphoid fevers
1,A02,137,Other salmonella infections
2,A03,36,Shigellosis
3,A04,7034,Other bacterial intestinal infections
4,A05,78,"Other bacterial foodborne intoxications, not e..."


In [13]:
categories.count()

icd10_category    1757
count             1757
description       1696
dtype: int64

In [29]:
categories[categories["description"].isna()]

,icd10_category,count,description
66,A90,13,None
67,A91,1,None
81,B10,76,None
117,B59,247,None
185,C4A,46,None
...,...,...,...
1725,Z68,56368,None
1726,Z69,3,None
1734,Z77,995,None
1735,Z78,10190,None


In [30]:
categories.loc[categories["description"].isna(), "description"] = categories.loc[categories["description"].isna(), "icd10_category"].apply(partial(get_description, lib=icd_cm))

In [32]:
categories.count()

icd10_category    1757
count             1757
description       1756
dtype: int64

In [33]:
categories[categories['description'].isna()]

,icd10_category,count,description
1025,NOD,30585,None


In [35]:
categories.loc[categories["icd10_category"].isin(["NOD"]), "description"] = "Unknown code"

In [36]:
categories.count()

icd10_category    1757
count             1757
description       1757
dtype: int64

In [37]:
categories.head()

,icd10_category,count,description
0,A01,13,Typhoid and paratyphoid fevers
1,A02,137,Other salmonella infections
2,A03,36,Shigellosis
3,A04,7034,Other bacterial intestinal infections
4,A05,78,"Other bacterial foodborne intoxications, not e..."


In [38]:
categories.to_csv("categories.tsv", sep="\t", index=False)

# Infer embeddings of descriptions

In [1]:
# !pip uninstall -y tensorflow tensorflow-gpu tensorflow-cpu tensorflow-metadata keras keras-preprocessing
# !pip install -U "transformers" "tokenizers" "sentencepiece" "protobuf"

In [2]:
# !pip check

In [3]:
import json
import pandas as pd
import numpy as np
import torch
from huggingface_hub import hf_hub_download
from safetensors import safe_open
from transformers import AutoTokenizer
from typing import List, Optional
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
from pickle import dump, load

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

MODELS = [
    "Qwen/Qwen3-235B-A22B-Instruct-2507",
    "yandex/YandexGPT-5-Lite-8B-instruct",
    "deepseek-ai/DeepSeek-V3"
]

/home/d.kornilov/work/Revealing-interconnections-between-diseases/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
categories = pd.read_csv("categories.tsv", sep="\t")

In [5]:
categories.head()

,icd10_category,count,description
0,A01,13,Typhoid and paratyphoid fevers
1,A02,137,Other salmonella infections
2,A03,36,Shigellosis
3,A04,7034,Other bacterial intestinal infections
4,A05,78,"Other bacterial foodborne intoxications, not e..."


In [6]:
def load_tok(model_id: str):
    try:
        return AutoTokenizer.from_pretrained(model_id, use_fast=True)
    except Exception:
        # slow SPM path (requires sentencepiece and protobuf compatible with it)
        return AutoTokenizer.from_pretrained(model_id, use_fast=False)

In [7]:
def get_embeddings(model_id: str, descriptions: pd.DataFrame, device: str="cpu", N_CORES_TO_USE: int=64, max_descriptions: Optional[int]=None):
    print("Processing with", model_id, "...")
    tok = load_tok(model_id)
    # AutoTokenizer.from_pretrained(model_id)
    
    # 1) Find which shard has the embedding weight
    print("Downloading index...")
    index_path = hf_hub_download(model_id, filename="model.safetensors.index.json")
    with open(index_path, "r") as f:
        index = json.load(f)
    weight_key = "model.embed_tokens.weight"
    shard_file = index["weight_map"][weight_key]
    
    # 2) Download only that shard and read the single tensor
    print("Downloading weights...")
    shard_path = hf_hub_download(model_id, filename=shard_file)
    with safe_open(shard_path, framework="pt", device="cpu") as f:
        W = f.get_tensor(weight_key)                  # (vocab_size, hidden_dim)
    
    # 3) Compute embeddings without a full model
    def infer_embedding_layer(args):
        print(args)
        idx, icd10_category, count, description = args
        ids = tok(description, return_tensors="pt")["input_ids"]  # CPU is fine; move to GPU if you want
        emb = torch.nn.functional.embedding(ids.to(device), W.to(device)).to("cpu")
        return icd10_category, description, emb

    embs = {}
    for args in tqdm(
        descriptions[:max_descriptions].itertuples(), total=len(descriptions[:max_descriptions]), desc="Computing embeddings..."
    ):
        icd10_category, description, emb = infer_embedding_layer(args)
        embs[icd10_category] = emb
        
    # with Pool(processes=min(N_CORES_TO_USE, cpu_count())) as pool:
    #     for icd10_category, description, embedding in tqdm(
    #         pool.imap_unordered(infer_embedding_layer, descriptions[:max_descriptions].itertuples(), chunksize=1),
    #         total=len(descriptions[:max_descriptions]),
    #         desc="Computing embeddings..."
    #     ):   
    #         embs[icd10_category] = embedding

    return embs

In [ ]:
# for model_id in tqdm([
#     "Qwen/Qwen3-235B-A22B-Instruct-2507",
#     "yandex/YandexGPT-5-Lite-8B-instruct",
#     "deepseek-ai/DeepSeek-V3",
# ]):
#     embs = get_embeddings(model_id=model_id, descriptions=categories, max_descriptions=5)
#     with open("embeddings/"+f"{model_id}.pkl".replace("/", "_"), "wb") as f:
#         dump(embs, f)

In [8]:
embs = {}

for model_id in MODELS:
    with open("embeddings/"+f"{model_id}.pkl".replace("/", "_"), "rb") as f: embs[model_id] = load(f)

/home/d.kornilov/work/Revealing-interconnections-between-diseases/.venv/lib/python3.10/site-packages/torch/storage.py:414: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  retu

In [9]:
embs['Qwen/Qwen3-235B-A22B-Instruct-2507']['A01'].shape

torch.Size([1, 10, 4096])

In [10]:
for model_id in embs:
    for icd10_category in embs[model_id]:
        embs[model_id][icd10_category] = embs[model_id][icd10_category].squeeze().max(axis=0).values #mean/max

In [14]:
embs['Qwen/Qwen3-235B-A22B-Instruct-2507']['A01'].shape

torch.Size([4096])

In [15]:
for model_id in embs:
    with open("embeddings/"+f"{model_id}_max.pkl".replace("/", "_"), "wb") as f:
        dump(embs, f)